# Badge 4: Data Lake Workshop: Lesson 8: Supercharging Development with Marketplace Data

Sonra OpenStreetMap Denver, variables, the DISTANCE_TO_MC UDF and the views over Sonra data.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE4_DLKW_ADMIN` (ACCOUNTADMIN), `BADGE4_DLKW_ROLE` (SYSADMIN) and `BADGE4_DLKW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.BADGE4_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE4_DLKW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE4_DLKW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE4_DLKW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
use schema MELS_SMOOTHIE_CHALLENGE_DB.TRAILS;
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 🥋 Shopping the Marketplace for GeoSpatial Data

> **Course:** NOTE: You will need to switch your role to ACCOUNTADMIN to get the share added to your trial. UI steps (DLKW_130, 131): Marketplace → search **Denver** → **OpenStreetMap Denver** (Sonra) → Get; Options: Database name **OpenStreetMap_Denver**; additional role SYSADMIN → Get.

In [ ]:
%%sql -r get_sonra_listing
-- SQL equivalent of Get. Needs the real ACCOUNTADMIN; the listing is Sonra's free OpenStreetMap Denver (GZSVZ3XR0R)
use role accountadmin;
call system$accept_legal_terms('DATA_EXCHANGE_LISTING', 'GZSVZ3XR0R');
create database OPENSTREETMAP_DENVER from listing 'GZSVZ3XR0R';
grant imported privileges on database OPENSTREETMAP_DENVER to role {{SYSADMIN_ROLE}};
-- switching roles drops the context, so restore it
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};
use schema MELS_SMOOTHIE_CHALLENGE_DB.TRAILS;

## Page: 🥋 Explore the Sonra Views

> **Course:** How many tables are available in the Sonra Denver share? Are there fewer tables than views? Of the views, about how many include "SHOP" in the view name? And about how many include "AMENITY"? (DLKW_135)

In [ ]:
%%sql -r sonra_objects
-- Derived: answers the questions above
select table_type, count(*) as objects,
       count_if(table_name ilike '%SHOP%') as with_shop_in_name,
       count_if(table_name ilike '%AMENITY%') as with_amenity_in_name
from OPENSTREETMAP_DENVER.INFORMATION_SCHEMA.TABLES
where table_schema = 'DENVER'
group by table_type;

## Page: 📓 Melanie's Café Location

Reading and map tools only (Google Maps, WKT Playground, geojson.io); nothing to run.

## Page: 🥋 Variables and Constants

In [ ]:
%%sql -r variables
-- Melanie's Location into a 2 Variables (mc for melanies cafe)
set mc_lng='-104.97300245114094';
set mc_lat='39.76471253574085';

--Confluence Park into a Variable (loc for location)
set loc_lng='-105.00840763333615'; 
set loc_lat='39.754141917497826';

--Test your variables to see if they work with the Makepoint function
select st_makepoint($mc_lng,$mc_lat) as melanies_cafe_point;
select st_makepoint($loc_lng,$loc_lat) as confluent_park_point;

--use the variables to calculate the distance from 
--Melanie's Cafe to Confluent Park
select st_distance(
        st_makepoint($mc_lng,$mc_lat)
        ,st_makepoint($loc_lng,$loc_lat)
        ) as mc_to_cp;

In [ ]:
%%sql -r constants
-- From screenshot DLKW_143 (second query): one set of constants and one set of variables
select st_distance(
        st_makepoint('-104.97300245114094','39.76471253574085')
        ,st_makepoint($loc_lng, $loc_lat)
        ) as mc_to_cp;

## Page: 🥋 Defining Our Own Function

> **Course:** Create a second Schema in Mel's Database and call it LOCATIONS. Make sure it is owned by SYSADMIN.

In [ ]:
%%sql -r create_locations_schema
-- Derived solution
use role {{SYSADMIN_ROLE}};
create schema MELS_SMOOTHIE_CHALLENGE_DB.LOCATIONS;
use schema MELS_SMOOTHIE_CHALLENGE_DB.LOCATIONS;

> ⚠️ **Audit note:** The course's first function block still contains the placeholder `<our function code will be here>`; the second still has `?` for the constants. Both run as given and report the result; the working function follows.

In [ ]:
%%sql -r udf_template
execute immediate 'begin
  execute immediate ''CREATE FUNCTION distance_to_mc(loc_lng number(38,32), loc_lat number(38,32))
  RETURNS FLOAT
  AS
  $$
    <our function code will be here>
  $$
  '';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

In [ ]:
%%sql -r udf_with_question_marks
execute immediate 'begin
  execute immediate ''CREATE OR REPLACE FUNCTION distance_to_mc(loc_lng number(38,32),loc_lat number(38,32))
  RETURNS FLOAT
  AS
  $$
   st_distance(
        st_makepoint(''''?'''',''''?'''')
        ,st_makepoint(loc_lng,loc_lat)
        )
  $$
  '';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

In [ ]:
%%sql -r udf_two_arguments
-- Derived solution: the question marks are Melanie's Cafe coordinates
CREATE OR REPLACE FUNCTION distance_to_mc(loc_lng number(38,32),loc_lat number(38,32))
  RETURNS FLOAT
  AS
  $$
   st_distance(
        st_makepoint('-104.97300245114094','39.76471253574085')
        ,st_makepoint(loc_lng,loc_lat)
        )
  $$
  ;

In [ ]:
%%sql -r test_udf
--Tivoli Center into the variables 
set tc_lng='-105.00532059763648'; 
set tc_lat='39.74548137398218';

select distance_to_mc($tc_lng,$tc_lat);

## Page: 🥋 Analyze Melanie's Competition

In [ ]:
%%sql -r juice_bars
select * 
from OPENSTREETMAP_DENVER.DENVER.V_OSM_DEN_AMENITY_SUSTENANCE
where 
    ((amenity in ('fast_food','cafe','restaurant','juice_bar'))
    and 
    (name ilike '%jamba%' or name ilike '%juice%'
     or name ilike '%superfruit%'))
 or 
    (cuisine like '%smoothie%' or cuisine like '%juice%');

> **Course:** 🎯 Convert the List into a View: COMPETITION, in the LOCATIONS schema, owned by SYSADMIN.

In [ ]:
%%sql -r create_competition
-- Derived solution
create view competition as
select * 
from OPENSTREETMAP_DENVER.DENVER.V_OSM_DEN_AMENITY_SUSTENANCE
where 
    ((amenity in ('fast_food','cafe','restaurant','juice_bar'))
    and 
    (name ilike '%jamba%' or name ilike '%juice%'
     or name ilike '%superfruit%'))
 or 
    (cuisine like '%smoothie%' or cuisine like '%juice%');

In [ ]:
%%sql -r closest_competitor
SELECT
 name
 ,cuisine
 , ST_DISTANCE(
    st_makepoint('-104.97300245114094','39.76471253574085')
    , coordinates
  ) AS distance_to_melanies
 ,*
FROM  competition
ORDER by distance_to_melanies;

In [ ]:
%%sql -r udf_geography
CREATE OR REPLACE FUNCTION distance_to_mc(lng_and_lat GEOGRAPHY)
  RETURNS FLOAT
  AS
  $$
   st_distance(
        st_makepoint('-104.97300245114094','39.76471253574085')
        ,lng_and_lat
        )
  $$
  ;

In [ ]:
%%sql -r closest_competitor_udf
SELECT
 name
 ,cuisine
 ,distance_to_mc(coordinates) AS distance_to_melanies
 ,*
FROM  competition
ORDER by distance_to_melanies;

## Page: 📓 Why Are There Two UDFs with the Same Name?

In [ ]:
%%sql -r overloaded_udf
-- Tattered Cover Bookstore McGregor Square
set tcb_lng='-104.9956203'; 
set tcb_lat='39.754874';

--this will run the first version of the UDF
select distance_to_mc($tcb_lng,$tcb_lat);

--this will run the second version of the UDF, bc it converts the coords 
--to a geography object before passing them into the function
select distance_to_mc(st_makepoint($tcb_lng,$tcb_lat));

--this will run the second version bc the Sonra Coordinates column
-- contains geography objects already
select name
, distance_to_mc(coordinates) as distance_to_melanies 
, ST_ASWKT(coordinates)
from OPENSTREETMAP_DENVER.DENVER.V_OSM_DEN_SHOP
where shop='books' 
and name like '%Tattered Cover%'
and addr_street like '%Wazee%';

## Page: 🎯 Analyze Potential Promotion Partners

> **Course:** 🎯 Create a View of Bike Shops in the Denver Data: DENVER_BIKE_SHOPS, in the LOCATIONS schema, owned by SYSADMIN; include DISTANCE_TO_MELANIES (DLKW_146).

In [ ]:
%%sql -r create_denver_bike_shops
-- From screenshot DLKW_146, blur filled in
create view DENVER_BIKE_SHOPS as
select name
 , ST_DISTANCE(
    st_makepoint('-104.97300245114094','39.76471253574085')
    , coordinates
  ) AS distance_to_melanies
  , coordinates
from OPENSTREETMAP_DENVER.DENVER.V_OSM_DEN_SHOP_OUTDOORS_AND_SPORT_VEHICLES
where shop = 'bicycle';

In [ ]:
%%sql -r bike_shop_count
select count(*) as bike_shops from denver_bike_shops;

## Page: 🤖 DORA DLKW08

### 🤖 DORA check DLKW08 goes here
The course has the learner run **DLKW08** (bike shop view distance calculation) at this point.
It is in `dora/dora_tests.ipynb`, section **DLKW08**, as a local check plus the exact DORA code.